In [112]:
import pandas as pd
import numpy as np
import json
import os
import re



"""
location信息
"""

location_df = pd.read_csv('Location.csv')
location_df.rename(columns={'Geneid':'Gene'}, inplace=True)

location_df

,Gene,Chr,Start,End,Strand,Length
0,NCU10129,NC_026501.1,1151,2878,-,1728
1,NCU09901,NC_026501.1,3178,6443,+,3266
2,NCU09903,NC_026501.1,9177,13334,+,4158
3,NCU11134,NC_026501.1,15930,16647,+,718
4,NCU09904,NC_026501.1,17441,19615,+,2175
...,...,...,...,...,...,...
10585,NCU16025,NC_026614.1,55087,57247,+,2161
10586,NCU16026,NC_026614.1,55087,56529,+,1443
10587,NCU16027,NC_026614.1,58608,58832,+,225
10588,NCU16028,NC_026614.1,59529,60281,+,753


In [113]:
"""
NCBI_gene_id信息
"""

ncbi_gene_id_df = pd.read_csv('NCBI_gene_id.csv', names=['NCBI_gene_id', 'Gene'])
ncbi_gene_id_df['Gene'] = ncbi_gene_id_df['Gene'].str.replace('ncr:','')
ncbi_gene_id_df['NCBI_gene_id'] = ncbi_gene_id_df['NCBI_gene_id'].str.replace('ncbi-geneid:','')


ncbi_gene_id_df

,NCBI_gene_id,Gene
0,3873166,NCU00003
1,3873167,NCU00004
2,3873168,NCU00005
3,3873169,NCU00006
4,3873170,NCU00007
...,...,...
10169,23569905,NCU17399
10170,23569906,NCU17402
10171,23569907,NCU17405
10172,23569908,NCU17408


In [114]:
"""
NCBI protein id 信息
"""

protein_id_df = pd.read_csv('Protein_id.csv', names=['Protein_id', 'Gene'])
protein_id_df['Gene'] = protein_id_df['Gene'].str.replace('ncr:','')
protein_id_df['Protein_id'] = protein_id_df['Protein_id'].str.replace('ncbi-proteinid:','')


protein_id_df

,Protein_id,Gene
0,XP_956990,NCU00003
1,XP_956991,NCU00004
2,XP_956992,NCU00005
3,XP_011393719,NCU00006
4,XP_956994,NCU00007
...,...,...
9753,XP_011395398,NCU17281
9754,XP_011395400,NCU17282
9755,XP_011395401,NCU17283
9756,XP_011395402,NCU17284


In [115]:
"""
uniport id 信息
"""

uniprot_id_df = pd.read_csv('Uniprot_id.csv', names=['Uniport_id', 'Gene'])
uniprot_id_df['Gene'] = uniprot_id_df['Gene'].str.replace('ncr:','')
uniprot_id_df['Uniport_id'] = uniprot_id_df['Uniport_id'].str.replace('up:','')

uniprot_id_df

,Uniport_id,Gene
0,Q7RYA2,NCU00003
1,Q7RYA1,NCU00004
2,Q7RYA0,NCU00005
3,U9W339,NCU00006
4,Q7RY98,NCU00007
...,...,...
9752,V5ILT8,NCU17281
9753,V5ILU0,NCU17282
9754,V5IL82,NCU17283
9755,V5IMX3,NCU17284


In [116]:
"""
合并location, NCBI_gene_id, protein_id, uniprot_id信息
"""

df_gene_info = pd.merge(location_df, ncbi_gene_id_df, on='Gene', how='outer')
df_gene_info = pd.merge(df_gene_info, protein_id_df, on='Gene', how='outer')
df_gene_info = pd.merge(df_gene_info, uniprot_id_df, on='Gene', how='outer')

# 删除Chr为NA的行
df_gene_info = df_gene_info.dropna(subset=['Chr'])

df_gene_info

,Gene,Chr,Start,End,Strand,Length,NCBI_gene_id,Protein_id,Uniport_id
0,NCU10129,NC_026501.1,1151.0,2878.0,-,1728.0,5847003,XP_001727976,A7UXG2
1,NCU09901,NC_026501.1,3178.0,6443.0,+,3266.0,3874163,XP_958016,Q7S0E6
2,NCU09903,NC_026501.1,9177.0,13334.0,+,4158.0,3874165,XP_958018,Q7S0E4
3,NCU11134,NC_026501.1,15930.0,16647.0,+,718.0,5847002,XP_001727977,A7UXG3
4,NCU09904,NC_026501.1,17441.0,19615.0,+,2175.0,3874166,XP_958019,Q7S0E3
...,...,...,...,...,...,...,...,...,...
10585,NCU16025,NC_026614.1,55087.0,57247.0,+,2161.0,23681580,YP_009126726,P37212
10586,NCU16026,NC_026614.1,55087.0,56529.0,+,1443.0,23681581,YP_009126727,M1RV38
10587,NCU16027,NC_026614.1,58608.0,58832.0,+,225.0,23681582,YP_009126728,Q12635
10588,NCU16028,NC_026614.1,59529.0,60281.0,+,753.0,23681583,YP_009126729,P00411


In [117]:
# Go.csv存在一对多情况
go_df = pd.read_csv('Go.csv')

# 将go_df的id列名改为Protein_id
go_df.rename(columns={'id':'Protein_id'}, inplace=True)

go_df['Protein_id'] = go_df['Protein_id'].str[:-2]

go_df

,Protein_id,GO,ONTOLOGY,GO_type
0,XP_001727958,"GO:0055114,GO:0016702","oxidation-reduction process,oxidoreductase act...","BP,MF"
1,XP_001727960,"GO:0005515,GO:0004190,GO:0006508","protein binding,aspartic-type endopeptidase ac...","MF,MF,BP"
2,XP_001727961,GO:0005515,protein binding,MF
3,XP_001727970,GO:0005515,protein binding,MF
4,XP_001727974,"GO:0004190,GO:0006508","aspartic-type endopeptidase activity,proteolysis","MF,BP"
...,...,...,...,...
5263,YP_009126725,"GO:0015986,GO:0015078,GO:0000276","ATP synthesis coupled proton transport,proton ...","BP,MF,CC"
5264,YP_009126726,"GO:0045263,GO:0015986,GO:0015078","proton-transporting ATP synthase complex, coup...","CC,BP,MF"
5265,YP_009126727,"GO:0045263,GO:0015986,GO:0015078,GO:0004519","proton-transporting ATP synthase complex, coup...","CC,BP,MF,MF"
5266,YP_009126728,"GO:0045263,GO:0015986,GO:0015078,GO:0033177,GO...","proton-transporting ATP synthase complex, coup...","CC,BP,MF,CC,BP,MF"


In [118]:
kegg_definition = pd.read_csv('KEGG_definition.csv')
kegg_definition.rename(columns={'Pathway':'KEGG_definition'}, inplace=True)

kegg_df = pd.read_csv('KEGG.csv', names=['KEGG', 'KEGG_definition'])

# 合并kegg_definition，kegg_df
kegg_merge_df = pd.merge(kegg_definition, kegg_df, on='KEGG_definition', how='inner')
kegg_merge_df = kegg_merge_df.fillna('')

kegg_merge_df = kegg_merge_df.groupby("Gene").agg({
    'KEGG_definition': lambda x: ','.join(x),
    'KEGG': lambda x: ','.join(x)
}).reset_index()

# 检测是否有重复行
# kegg_merge_df[kegg_merge_df.duplicated(['Gene'], keep=False)]

kegg_merge_df

,Gene,KEGG_definition,KEGG
0,NCU00008,"Metabolic pathways,Sphingolipid metabolism","ncr01100,ncr00600"
1,NCU00018,Protein processing in endoplasmic reticulum,ncr04141
2,NCU00034,"Biosynthesis of secondary metabolites,Terpenoi...","ncr01110,ncr00900"
3,NCU00035,"Metabolic pathways,Glycerolipid metabolism","ncr01100,ncr00561"
4,NCU00037,mRNA surveillance pathway,ncr03015
...,...,...,...
2224,NCU17399,"Ribosome,Ribosome biogenesis in eukaryotes","ncr03010,ncr03008"
2225,NCU17402,"Ribosome,Ribosome biogenesis in eukaryotes","ncr03010,ncr03008"
2226,NCU17405,"Ribosome,Ribosome biogenesis in eukaryotes","ncr03010,ncr03008"
2227,NCU17408,"Ribosome,Ribosome biogenesis in eukaryotes","ncr03010,ncr03008"


In [119]:
"""
合并go_df, kegg_merge_df
"""

df_gene_info = pd.merge(df_gene_info, go_df, on='Protein_id', how='outer')
df_gene_info = pd.merge(df_gene_info, kegg_merge_df, on='Gene', how='outer')


# 删除Chr这一列值为NaN的行
df_gene_info = df_gene_info.dropna(subset=['Chr'])

df_gene_info

,Gene,Chr,Start,End,Strand,Length,NCBI_gene_id,Protein_id,Uniport_id,GO,ONTOLOGY,GO_type,KEGG_definition,KEGG
0,NCU10129,NC_026501.1,1151.0,2878.0,-,1728.0,5847003,XP_001727976,A7UXG2,NaN,NaN,NaN,NaN,NaN
1,NCU09901,NC_026501.1,3178.0,6443.0,+,3266.0,3874163,XP_958016,Q7S0E6,GO:0003676,nucleic acid binding,MF,"Nucleocytoplasmic transport,mRNA surveillance ...","ncr03013,ncr03015"
2,NCU09903,NC_026501.1,9177.0,13334.0,+,4158.0,3874165,XP_958018,Q7S0E4,"GO:0016887,GO:0005524","ATPase activity,ATP binding","MF,MF",NaN,NaN
3,NCU11134,NC_026501.1,15930.0,16647.0,+,718.0,5847002,XP_001727977,A7UXG3,NaN,NaN,NaN,NaN,NaN
4,NCU09904,NC_026501.1,17441.0,19615.0,+,2175.0,3874166,XP_958019,Q7S0E3,"GO:0005975,GO:0004553","carbohydrate metabolic process,hydrolase activ...","BP,MF",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10585,NCU16024,NC_026614.1,54186.0,54350.0,+,165.0,23681579,YP_009126725,Q08656,"GO:0015986,GO:0015078,GO:0000276","ATP synthesis coupled proton transport,proton ...","BP,MF,CC","Metabolic pathways,Oxidative phosphorylation","ncr01100,ncr00190"
10586,NCU16025,NC_026614.1,55087.0,57247.0,+,2161.0,23681580,YP_009126726,P37212,"GO:0045263,GO:0015986,GO:0015078","proton-transporting ATP synthase complex, coup...","CC,BP,MF","Metabolic pathways,Oxidative phosphorylation","ncr01100,ncr00190"
10587,NCU16026,NC_026614.1,55087.0,56529.0,+,1443.0,23681581,YP_009126727,M1RV38,"GO:0045263,GO:0015986,GO:0015078,GO:0004519","proton-transporting ATP synthase complex, coup...","CC,BP,MF,MF",NaN,NaN
10588,NCU16027,NC_026614.1,58608.0,58832.0,+,225.0,23681582,YP_009126728,Q12635,"GO:0045263,GO:0015986,GO:0015078,GO:0033177,GO...","proton-transporting ATP synthase complex, coup...","CC,BP,MF,CC,BP,MF","Metabolic pathways,Oxidative phosphorylation","ncr01100,ncr00190"


In [120]:
"""
KO id 信息
"""

kunm_df = pd.read_csv('Knum.csv', names=['Protein_id', 'Knum'])
kunm_df = kunm_df.groupby("Protein_id")["Knum"].apply(lambda x: ','.join(x)).reset_index()

kunm_df['Protein_id'] = kunm_df['Protein_id'].str[:-2]

kunm_df

,Protein_id,Knum
0,XP_001727958,K17842
1,XP_001727960,K06653
2,XP_001727961,K06653
3,XP_001727974,"K01379,K01381"
4,XP_001727981,K18064
...,...,...
4399,YP_009126721,K03881
4400,YP_009126726,K02126
4401,YP_009126727,K02126
4402,YP_009126728,"K02128,K03880"


In [121]:
"""
EC number
"""

ec_number_df = pd.read_csv('EC_number.csv', names=['Protein_id', 'EC_number'])
ec_number_df = ec_number_df.groupby("Protein_id")["EC_number"].apply(lambda x: ','.join(x)).reset_index()

ec_number_df['Protein_id'] = ec_number_df['Protein_id'].str[:-2]

ec_number_df

,Protein_id,EC_number
0,XP_001727958,1.13.11.59
1,XP_001727974,"3.4.23.25,3.4.23.5"
2,XP_001727998,3.1.4.11
3,XP_001728014,3.6.1.23
4,XP_001728015,2.7.1.35
...,...,...
1957,YP_009126712,1.6.5.3
1958,YP_009126717,1.9.3.1
1959,YP_009126719,1.6.5.3
1960,YP_009126721,1.6.5.3


In [122]:
"""
功能描述信息
"""

description_df = pd.read_csv('Description.csv', names=['Protein_id', 'Description'])

description_df['Protein_id'] = description_df['Protein_id'].str[:-2]

description_df

,Protein_id,Description
0,XP_001727958,Retinal pigment epithelial membrane protein
1,XP_001727959,Domain of unknown function (DUF1989)
2,XP_001727960,Glycerophosphoryl diester phosphodiesterase fa...
3,XP_001727961,Domain that may be involved in binding ubiquit...
4,XP_001727966,-
...,...,...
10200,YP_009126722,LAGLIDADG endonuclease
10201,YP_009126726,Mitochondrial membrane ATP synthase (F(1)F(0) ...
10202,YP_009126727,Mitochondrial membrane ATP synthase (F(1)F(0) ...
10203,YP_009126728,Belongs to the ATPase C chain family(atp9)


In [123]:
pfam_df = pd.read_csv('PFAM.csv', names=['Protein_id', 'Pfam'])
pfam_df = pfam_df.groupby("Protein_id")["Pfam"].apply(lambda x: ','.join(x)).reset_index()

pfam_df['Protein_id'] = pfam_df['Protein_id'].str[:-2]

pfam_df

,Protein_id,Pfam
0,XP_001727958,RPE65
1,XP_001727959,DUF1989
2,XP_001727960,"Ank_2,Ank_4,GDPD,SPX"
3,XP_001727961,CUE
4,XP_001727969,Clr5
...,...,...
7561,YP_009126722,LAGLIDADG_1
7562,YP_009126726,ATP-synt_A
7563,YP_009126727,ATP-synt_A
7564,YP_009126728,ATP-synt_C


In [124]:
"""
利用ProCAST对mt的蛋白序列文件进行注释后，得到Cazy表
"""

cazy_df = pd.read_csv('./Cazy.csv')
cazy_df = cazy_df.iloc[:, :2]

# 去掉无关数据
cazy_df['subject_id'] = cazy_df['subject_id'].str.replace(r'\w+\.1', '', regex=True)

# 去掉多余空格
cazy_df['subject_id'] = cazy_df['subject_id'].str.lstrip()

# 将cazy_df第一列列名改为Protein_id，第二列列名改为Cazy
cazy_df.rename(columns={'query_id':'Protein_id', 'subject_id':'Cazy'}, inplace=True)

# 将第一列的Protein_id相同的行合并，Cazy列的值用逗号隔开
cazy_df = cazy_df.groupby("Protein_id")["Cazy"].apply(lambda x: ','.join(x)).reset_index()

cazy_df['Protein_id'] = cazy_df['Protein_id'].str[:-2]

display(cazy_df)

,Protein_id,Cazy
0,XP_001728006,GH71: Glycoside Hydrolases (GHs) CBM24: Carboh...
1,XP_001728007,GH71: Glycoside Hydrolases (GHs) CBM24: Carboh...
2,XP_001728028,GT22: GlycosylTransferases (GTs)
3,XP_001728031,GH93: Glycoside Hydrolases (GHs)
4,XP_001728032,GH93: Glycoside Hydrolases (GHs)
...,...,...
797,XP_965749,GT32: GlycosylTransferases (GTs)
798,XP_965769,GH76: Glycoside Hydrolases (GHs)
799,XP_965782,GH35: Glycoside Hydrolases (GHs)
800,XP_965803,GT0: GlycosylTransferases (GTs)


In [125]:
"""
合并kunm_df, ec_number_df, description_df, pfam_df, cazy_df
"""

df_gene_info = pd.merge(df_gene_info, kunm_df, on='Protein_id', how='outer')
df_gene_info = pd.merge(df_gene_info, ec_number_df, on='Protein_id', how='outer')
df_gene_info = pd.merge(df_gene_info, description_df, on='Protein_id', how='outer')
df_gene_info = pd.merge(df_gene_info, pfam_df, on='Protein_id', how='outer')
df_gene_info = pd.merge(df_gene_info, cazy_df, on='Protein_id', how='outer')

# 删除掉Chr为NaN的行
df_gene_info = df_gene_info.dropna(subset=['Chr'])

df_gene_info


,Gene,Chr,Start,End,Strand,Length,NCBI_gene_id,Protein_id,Uniport_id,GO,ONTOLOGY,GO_type,KEGG_definition,KEGG,Knum,EC_number,Description,Pfam,Cazy
0,NCU10129,NC_026501.1,1151.0,2878.0,-,1728.0,5847003,XP_001727976,A7UXG2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN
1,NCU09901,NC_026501.1,3178.0,6443.0,+,3266.0,3874163,XP_958016,Q7S0E6,GO:0003676,nucleic acid binding,MF,"Nucleocytoplasmic transport,mRNA surveillance ...","ncr03013,ncr03015",K14325,NaN,Pfam:RRM_6,RRM_1,GH0: Glycoside Hydrolases (GHs)
2,NCU09903,NC_026501.1,9177.0,13334.0,+,4158.0,3874165,XP_958018,Q7S0E4,"GO:0016887,GO:0005524","ATPase activity,ATP binding","MF,MF",NaN,NaN,K03235,NaN,Chromatin organization modifier domain(NEW1),"ABC_tran,Chromo",NaN
3,NCU11134,NC_026501.1,15930.0,16647.0,+,718.0,5847002,XP_001727977,A7UXG3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN
4,NCU09904,NC_026501.1,17441.0,19615.0,+,2175.0,3874166,XP_958019,Q7S0E3,"GO:0005975,GO:0004553","carbohydrate metabolic process,hydrolase activ...","BP,MF",NaN,NaN,NaN,NaN,Glycosyl hydrolases family 16,Glyco_hydro_16,GH16_4: Glycoside Hydrolases (GHs)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10585,NCU16024,NC_026614.1,54186.0,54350.0,+,165.0,23681579,YP_009126725,Q08656,"GO:0015986,GO:0015078,GO:0000276","ATP synthesis coupled proton transport,proton ...","BP,MF,CC","Metabolic pathways,Oxidative phosphorylation","ncr01100,ncr00190",NaN,NaN,NaN,NaN,NaN
10586,NCU16025,NC_026614.1,55087.0,57247.0,+,2161.0,23681580,YP_009126726,P37212,"GO:0045263,GO:0015986,GO:0015078","proton-transporting ATP synthase complex, coup...","CC,BP,MF","Metabolic pathways,Oxidative phosphorylation","ncr01100,ncr00190",K02126,NaN,Mitochondrial membrane ATP synthase (F(1)F(0) ...,ATP-synt_A,NaN
10587,NCU16026,NC_026614.1,55087.0,56529.0,+,1443.0,23681581,YP_009126727,M1RV38,"GO:0045263,GO:0015986,GO:0015078,GO:0004519","proton-transporting ATP synthase complex, coup...","CC,BP,MF,MF",NaN,NaN,K02126,NaN,Mitochondrial membrane ATP synthase (F(1)F(0) ...,ATP-synt_A,NaN
10588,NCU16027,NC_026614.1,58608.0,58832.0,+,225.0,23681582,YP_009126728,Q12635,"GO:0045263,GO:0015986,GO:0015078,GO:0033177,GO...","proton-transporting ATP synthase complex, coup...","CC,BP,MF,CC,BP,MF","Metabolic pathways,Oxidative phosphorylation","ncr01100,ncr00190","K02128,K03880",1.6.5.3,Belongs to the ATPase C chain family(atp9),ATP-synt_C,NaN


In [126]:
amino_acid_sequence_df = pd.read_csv('Amino_acid_sequence.csv')
amino_acid_sequence_df.rename(columns={'Protein ID':'Protein_id','Sequence':'Amino_acid_sequence'}, inplace=True)

amino_acid_sequence_df['Protein_id'] = amino_acid_sequence_df['Protein_id'].str[:-2]
amino_acid_sequence_df

,Protein_id,Amino_acid_sequence
0,XP_001727958,MSPHEVIGTVPKNSTTFRTQADEHDDHEEALQNLRTGKYEDWPNEA...
1,XP_001727959,MTTPEQDGAARLAARKPIPKPVPAYLPTAGSPLSVDKNLYSTIQQA...
2,XP_001727960,MKFGKQIQKRQLEVPEYAASFVNYKALKKLIKKLSATPILPPQTDL...
3,XP_001727961,MSAPTDSNVSKHATVEPAPESPTTARPFELDDDEAQENVPLGTDTT...
4,XP_001727966,MKLEQSSTPKITGLIRILTLTSEEENRAQSDQVQAIIRDRLRNLNY...
...,...,...
10807,YP_009126725,MPQLVPFYFVNEITFTFVIITLMVYILSKYILPRFVRLFLSRTFIS...
10808,YP_009126726,MSTLSFNNISTEVLSPLNQFEIRDLLSIDALGNLHISITNIGFYLT...
10809,YP_009126727,MSTLSFNNISTEVLSPLNQFEIRDLLSIDALGNLHISITNIGFYLT...
10810,YP_009126728,MIQVAKIIGTGLATTGLIGAGIGIGVVFGSLIIGVSRNPSLKSQLF...


In [127]:
nucleic_acid_df = pd.read_csv('Nucleic_acid_sequence.csv')
nucleic_acid_df.rename(columns={'gene':'Gene','sequence':'Nucleic_acid_sequence'}, inplace=True)

nucleic_acid_df

,Gene,Nucleic_acid_sequence
0,NCU10129,ATGCCTCTCTTCTCCCGCCACGCCGAGCCTGAGCCCGCCCCGGCCC...
1,NCU09901,ATGGCCTCCCGCTCGAGGTCTCGGTCGGCCGACCGCTACGATGACG...
2,NCU09903,ATGACAGTCCCTATCATGGTCGCTGATGGCGCCGCGCCGCCCGTCA...
3,NCU11134,ATGGTTGTTCAAGCTCCCATGCTACAAGAGCCCTCACTGGTTCCAC...
4,NCU09904,ATGGACCACAGACACACCATGGACAACGACCACCGCGAACATCTCG...
...,...,...
10807,NCU16024,ATGCCTCAATTAGTTCCATTTTATTTTGTTAATGAAATAACTTTTA...
10808,NCU16025,ATGAGTACTTTAAGCTTTAATAATATATCGACAGAAGTCCTTAGCC...
10809,NCU16026,ATGAGTACTTTAAGCTTTAATAATATATCGACAGAAGTCCTTAGCC...
10810,NCU16027,ATGATACAAGTAGCTAAAATAATAGGAACAGGGCTAGCTACCACAG...


In [128]:
# 将gene_information_df与amino_acid_sequence_df合并
gene_information_df = pd.merge(df_gene_info, amino_acid_sequence_df, on='Protein_id', how='left')

# 将gene_information_df与nucleic_acid_df合并
gene_information_df = pd.merge(gene_information_df, nucleic_acid_df, on='Gene', how='left')

# 删除Gene列中的重复值
gene_information_df.drop_duplicates(subset=['Gene'], keep='first', inplace=True)

gene_information_df.to_csv('gene_information.csv', index=False)

In [129]:
df_change = pd.read_csv('gene_information.csv')

df_dict = df_change.fillna('null').to_dict(orient='records')

In [130]:
df_dict

[{'Gene': 'NCU10129',
  'Chr': 'NC_026501.1',
  'Start': 1151.0,
  'End': 2878.0,
  'Strand': '-',
  'Length': 1728.0,
  'NCBI_gene_id': 5847003.0,
  'Protein_id': 'XP_001727976',
  'Uniport_id': 'A7UXG2',
  'GO': 'null',
  'ONTOLOGY': 'null',
  'GO_type': 'null',
  'KEGG_definition': 'null',
  'KEGG': 'null',
  'Knum': 'null',
  'EC_number': 'null',
  'Description': '-',
  'Pfam': 'null',
  'Cazy': 'null',
  'Amino_acid_sequence': 'MPLFSRHAEPEPAPAPVYQEPAPKRHSLFGGSHHRSPSPGSPTYSSASDRNRSTSRDSGHGTQRKGSLLSRTFGNGTSTSHDVDPSIMQARERVMMAEAAERDADRALMAARESVRQAREHVRALELEAQEEARRAKIKQYHAKEVSKRGKQLGRHGI',
  'Nucleic_acid_sequence': 'ATGCCTCTCTTCTCCCGCCACGCCGAGCCTGAGCCCGCCCCGGCCCCCGTCTACCAGGAACCTGCTCCCAAGCGCCACTCCCTCTTCGGCGGTTCCCACCACCGCTCGCCCTCTCCCGGGTCGCCCACCTACTCCTCCGCATCGGACCGCAACCGCTCCACCAGCCGAGACTCCGGACACGGAACTCAGCGCAAGGGCAGCCTGTTGTCTCGCACCTTCGGCAACGGTACCTCGACCAGCCACGACGTCGACCCGTCCATCATGCAGGCTCGCGAGCGCGTTATGATGGCCGAGGCTGCCGAACGCGATGCGGACCGAGCGCTGATGGCTGCCAGGGAGAGCGTAAGGCAGGCTCGGGAGCATGTGAGGGCGTT

In [131]:
output_dict = {}

for item in df_dict:
    Gene = 'KEY:' + item['Gene']
    output_dict[Gene] = {
        'Gene': item['Gene'],
        'NCBI_gene_id': int(item['NCBI_gene_id']) if item['NCBI_gene_id'] != 'null' else None,
        'Protein_id': item['Protein_id'],
        'Uniport_id': item['Uniport_id'],
        'Knum': str(item['Knum']),
        'Description': item['Description'],
        'Cazy': item['Cazy'],
        'PFAM': item['Pfam'],
        'Chr': item['Chr'],
        'Start': int(item['Start']) if item['Start'] != 'null' else None,
        'End': int(item['End']) if item['End'] != 'null' else None,
        'Strand': item['Strand'],
        'Length': int(item['Length']) if item['Length'] != 'null' else None,
        'Nucleic_acid_sequence': item['Nucleic_acid_sequence'],
        'Amino_acid_sequence': item['Amino_acid_sequence'],
        'GO_information': {
            'GO': item['GO'],
            'ONTOLOGY': item['ONTOLOGY'],
            'GO_type': item['GO_type'],
        },
        'KEGG_information': {
            'KEGG': str(item['KEGG']),
            'KEGG_definition': str(item['KEGG_definition']),
        },
        'EC_number': str(item['EC_number']),
    }

In [ ]:
for key, value in output_dict.items():
    GO = value['GO_information']['GO'].split(',')
    ONTOLOGY = value['GO_information']['ONTOLOGY'].split(',')
    GO_type = value['GO_information']['GO_type'].split(',')
    GO_information = []
    for i in range(len(GO)):
        GO_information.append({
            'GO': GO[i],
            'ONTOLOGY': ONTOLOGY[i],
            'GO_type': GO_type[i]
        })
    output_dict[key]['GO_information'] = GO_information

for key, value in output_dict.items():
    KEGG = value['KEGG_information']['KEGG'].split(',')
    KEGG_definition = value['KEGG_information']['KEGG_definition'].split(',')
    KEGG_information = []
    for i in range(len(KEGG)):
        KEGG_information.append({
            'KEGG': KEGG[i],
            'KEGG_definition': KEGG_definition[i],
        })
    output_dict[key]['KEGG_information'] = KEGG_information

output_dict

In [133]:
with open('gene_information.json', 'w') as f:
    json.dump(output_dict, f, indent=4)